# Surrogates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
rcupd = {
    'figure.figsize': (5, 4),
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': 'cm',
    'font.size': 12,
}
plt.rcParams.update(rcupd)

In [ ]:
data_files = [
    '2025-11-04/RERTR5_V6018G.csv',
    '2025-11-04/RERTR12_L1P755.csv',
]

# Convenience functions

In [ ]:
def lin_coef_cept(mod):
    print(
        ' coeffs: ',
        mod.coef_, '\n',
        'intercept: ',
        mod.intercept_
    )

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

def mod_metrics(mod, X_test, y_test):
    y_pred = mod.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(
        ' R2: ', r2, '\n',
        'RMSE: ', rmse, '\n',
        'MAE: ', mae
    )

In [ ]:
def pred_vs_actual(mod, X_test, y_test, tt):
    y_pred = mod.predict(X_test)

    plt.figure(figsize=(5,4))
    plt.rcParams.update({'font.size': 16})

    plt.scatter(y_test, y_pred, s=15)

    minv = int(min(min(y_test), min(y_pred)))
    maxv = int(max(max(y_test), max(y_pred)))
    val = list(range(minv, maxv))
    
    plt.plot(val, val, color='k', ls='--', label='y=x')

    plt.title(tt)
    plt.xlabel(r'Test data (swelling \%)')
    plt.ylabel(r'Surrogate pred. (swelling \%)')
    plt.legend()
    plt.tight_layout()
    plt.show()
    #plt.savefig(f'{tt}.pdf')

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

def load_data(fileName):
    jar = pd.read_csv(fileName)
    # vResol issue
    jar['vResol'] = jar['vResol'] * 1e18

    X = jar.iloc[:, 1:-2].to_numpy()
    y = jar.iloc[:, -2].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=19
    )

    xscaler = MinMaxScaler()
    xscaler.fit(X_train)

    X_train = xscaler.transform(X_train)
    X_test = xscaler.transform(X_test)

    return X_train, X_test, y_train, y_test

# Linear (OLS)

In [ ]:
from sklearn import linear_model

In [ ]:
for f in data_files:
    X_train, X_test, y_train, y_test = load_data(f)
    
    reg_lin = linear_model.LinearRegression().fit(X_train, y_train)
    
    lin_coef_cept(reg_lin)
    mod_metrics(reg_lin, X_test, y_test)
    pred_vs_actual(reg_lin, X_test, y_test, r'OLS for \textit{' + f[11:-4] + '}')

# Lasso

In [ ]:
for f in data_files:
    X_train, X_test, y_train, y_test = load_data(f)
    
    reg_lin = linear_model.Lasso(alpha=0.1).fit(X_train, y_train)
    
    lin_coef_cept(reg_lin)
    mod_metrics(reg_lin, X_test, y_test)
    pred_vs_actual(reg_lin, X_test, y_test, r'Lasso for \textit{' + f[11:-4] + '}')

# NN

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
regs_nn = []

for f in data_files:
    X_train, X_test, y_train, y_test = load_data(f)
    
    reg_nn = MLPRegressor(
        hidden_layer_sizes=(250, 250, 250, 250),
        alpha=0,
        random_state=37,
        max_iter=5000,
        tol=0.1
    ).fit(X_train, y_train)

    mod_metrics(reg_nn, X_test, y_test)
    pred_vs_actual(reg_nn, X_test, y_test, r'NN for \textit{' + f[11:-4] + '}')

    regs_nn.append(reg_nn)

In [ ]:
import pickle

with open('nn_surrogates.pkl', 'wb') as f:
    pickle.dump(regs_nn, f)

# SVR

In [ ]:
from sklearn.svm import SVR

In [ ]:
for f in data_files:
    X_train, X_test, y_train, y_test = load_data(f)
    
    reg_svr = SVR(kernel='rbf').fit(X_train, y_train)
    
    mod_metrics(reg_svr, X_test, y_test)
    pred_vs_actual(reg_svr, X_test, y_test, r'SVR for \textit{' + f[11:-4] + '}')

# GP

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    DotProduct, WhiteKernel, RBF, ConstantKernel, ExpSineSquared, Matern
)

In [ ]:
regs_gp = []

for f in data_files:
    X_train, X_test, y_train, y_test = load_data(f)
    
    tX_train, _, ty_train, _ = train_test_split(
        X_train, y_train, test_size=0.7, random_state=127
    )
    print(tX_train.shape)

    #kern = DotProduct(sigma_0_bounds=(1e-5, 1e6))
    #kern = ConstantKernel() * RBF()
    kern = ConstantKernel() * Matern()
    reg_gp = GaussianProcessRegressor(
        kernel=kern,
        alpha=1e-3,
        n_restarts_optimizer=9,
        random_state=42
    )
    reg_gp.fit(tX_train, ty_train)
    print(reg_gp.kernel_)

    mod_metrics(reg_gp, X_test, y_test)
    pred_vs_actual(reg_gp, X_test, y_test, r'GP for \textit{' + f[11:-4] + '}')

    regs_gp.append(reg_gp)

In [ ]:
import pickle

with open('gp_surrogates.pkl', 'wb') as f:
    pickle.dump(regs_gp, f)